# 1. Duplicate Data & Safe Type Casting

This notebook covers:
1. Identifying and resolving **Exact Duplicates** vs. **Subset (Semantic) Duplicates**.
2. Fixing incorrect data types and applying safe numeric/datetime conversions using `errors='coerce'`.
3. Practical rules to prevent duplicate-based **Data Leakage**.

In [23]:
import numpy as np
import pandas as pd

# Creating a messy raw dataset with duplicate records and broken data types
raw_data = {
    'Transaction_ID': [1001, 1002, 1003, 1001, 1004, 1002, 1005],
    'Customer_Name': ['Alice', 'Bob', 'Charlie', 'Alice', 'David', 'Bob', 'Eve'],
    'Age': ['25', '30', 'unknown', '25', '45', '30', '29'],
    'Amount_Paid': ['$150.00', '$200.50', '$99.00', '$150.00', '$320.00', '$210.00', 'free'],
    'City': ['Hyderabad', 'Bengaluru', 'Delhi', 'Hyderabad', 'Mumbai', 'Bangalore', 'Chennai']
}

df = pd.DataFrame(raw_data)
print("=== RAW MESSY DATASET ===")
display(df)
print("\nData Types:")
print(df.dtypes)

=== RAW MESSY DATASET ===


,Transaction_ID,Customer_Name,Age,Amount_Paid,City
0,1001,Alice,25,$150.00,Hyderabad
1,1002,Bob,30,$200.50,Bengaluru
2,1003,Charlie,unknown,$99.00,Delhi
3,1001,Alice,25,$150.00,Hyderabad
4,1004,David,45,$320.00,Mumbai
5,1002,Bob,30,$210.00,Bangalore
6,1005,Eve,29,free,Chennai



Data Types:
Transaction_ID     int64
Customer_Name     object
Age               object
Amount_Paid       object
City              object
dtype: object


---
## Part 1: Duplicate Detection & Removal

- **Exact Duplicates:** Every single column value is identical across rows.
- **Subset / Semantic Duplicates:** A primary key or business ID (e.g., `Transaction_ID`) matches, but non-key columns may have slight updates.
- **Leakage Rule:** Deduplication must be performed **BEFORE** the train/test split.

In [36]:
df.duplicated()

0    False
1    False
2    False
3     True
4    False
5    False
6    False
dtype: bool

In [37]:
new_df=df.copy()
new_df = new_df.drop_duplicates(keep='first')
display(new_df)

,Transaction_ID,Customer_Name,Age,Amount_Paid,City
0,1001,Alice,25,$150.00,Hyderabad
1,1002,Bob,30,$200.50,Bengaluru
2,1003,Charlie,unknown,$99.00,Delhi
4,1004,David,45,$320.00,Mumbai
5,1002,Bob,30,$210.00,Bangalore
6,1005,Eve,29,free,Chennai


In [39]:
display(new_df[new_df.duplicated(subset=['Transaction_ID'], keep=False)])

,Transaction_ID,Customer_Name,Age,Amount_Paid,City
1,1002,Bob,30,$200.50,Bengaluru
5,1002,Bob,30,$210.00,Bangalore


In [44]:
new_df = new_df.drop_duplicates(subset=['Transaction_ID'], keep='first')
display(new_df)

,Transaction_ID,Customer_Name,Age,Amount_Paid,City
0,1001,Alice,25,$150.00,Hyderabad
1,1002,Bob,30,$200.50,Bengaluru
2,1003,Charlie,unknown,$99.00,Delhi
4,1004,David,45,$320.00,Mumbai
6,1005,Eve,29,free,Chennai


In [42]:
# 1. Check for exact full-row duplicates
print("Exact duplicate rows count:", df.duplicated().sum())

# View duplicated rows
print("\nIdentical rows:")
display(df[df.duplicated(keep=False)])

# 2. Drop exact duplicates (removes row 3, which is an exact clone of row 0)
df_clean = df.drop_duplicates(keep='first').copy()

# 3. Handle subset duplicates
# Row 1 and Row 5 have the same Transaction_ID 1002, but different Amount_Paid and City
print("\nTransactions with duplicate IDs:")
display(df_clean[df_clean.duplicated(subset=['Transaction_ID'], keep=False)])

# Keep the latest record ('last')
df_clean = df_clean.drop_duplicates(subset=['Transaction_ID'], keep='last').reset_index(drop=True)

print("\n=== DATASET AFTER DEDUPLICATION ===")
display(df_clean)

Exact duplicate rows count: 1

Identical rows:


,Transaction_ID,Customer_Name,Age,Amount_Paid,City
0,1001,Alice,25,$150.00,Hyderabad
3,1001,Alice,25,$150.00,Hyderabad



Transactions with duplicate IDs:


,Transaction_ID,Customer_Name,Age,Amount_Paid,City
1,1002,Bob,30,$200.50,Bengaluru
5,1002,Bob,30,$210.00,Bangalore



=== DATASET AFTER DEDUPLICATION ===


,Transaction_ID,Customer_Name,Age,Amount_Paid,City
0,1001,Alice,25,$150.00,Hyderabad
1,1003,Charlie,unknown,$99.00,Delhi
2,1004,David,45,$320.00,Mumbai
3,1002,Bob,30,$210.00,Bangalore
4,1005,Eve,29,free,Chennai
